# Loading netCDF files compressed with HDF5 compression filters

NetCDF4 is built on top of the HDF5 file format. 
A feature of HDF5 is "[filter plugins](https://github.com/HDFGroup/hdf5_plugins/blob/master/docs/RegisteredFilterPlugins.md)".
With these filters it is possible to compress data using alternative compression algorithms.

For our research, lossy compression can be very useful. 
There already is a high degree of uncertainty in the machine learning predictions, so we do not need (for example) 0.01 K accuracy in the air temperature data.

By using an error-controlled lossy compressor (e.g. [SZ3](https://ieeexplore.ieee.org/document/8622520) or SZ) we can siginificantly reduce the amount of storage required for our input data (i.e. global ERA5).

The netcdf-c library sadly does not support HDF5 filters (yet), so loading the data is slightly less straightforward.

The script `/download_scripts/download_compress_ERA5.py` can be used to download the required global ERA5 imput data and compress it using SZ3/SZ. The required storage is 725 GB.

In [1]:
import hdf5plugin  # Must be imported for the plugins to be registered
import xarray as xr
from pathlib import Path

It is useful to first check if the data we want to load is compressed using HDF5 filters.
For this we can use the h5encoded function:

In [2]:
from excited_workflow.utils import h5encoded

In [3]:
era5files = list(Path("/data/volume_2/hourly_global_era5").glob("*.nc")) 
h5encoded(era5files)

True

To actually read the files, we need to specify an alterinative engine to xarray:

In [4]:
ds = xr.open_mfdataset(era5files, engine="h5netcdf")

As you can see, global data is available from 2000-2020:

In [5]:
ds

<xarray.Dataset> Size: 6TB
Dimensions:    (longitude: 1440, latitude: 721, time: 175320)
Coordinates:
  * longitude  (longitude) float32 6kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8
  * latitude   (latitude) float32 3kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * time       (time) datetime64[ns] 1MB 2000-01-01 ... 2019-12-31T23:00:00
Data variables:
    d2m        (time, latitude, longitude) float32 728GB dask.array<chunksize=(2, 721, 1440), meta=np.ndarray>
    mslhf      (time, latitude, longitude) float32 728GB dask.array<chunksize=(2, 721, 1440), meta=np.ndarray>
    msshf      (time, latitude, longitude) float32 728GB dask.array<chunksize=(2, 721, 1440), meta=np.ndarray>
    sp         (time, latitude, longitude) float32 728GB dask.array<chunksize=(2, 721, 1440), meta=np.ndarray>
    ssr        (time, latitude, longitude) float32 728GB dask.array<chunksize=(2, 721, 1440), meta=np.ndarray>
    str        (time, latitude, longitude) float32 728GB dask.array<chunksize=(2, 721, 1440), meta=np.ndarray>
    t2m        (time, latitude, longitude) float32 728GB dask.array<chunksize=(2, 721, 1440), meta=np.ndarray>
    tp         (time, latitude, longitude) float32 728GB dask.array<chunksize=(2, 721, 1440), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.6
    history:      2024-05-08 02:39:23 GMT by grib_to_netcdf-2.28.1: /opt/ecmw...

As usual, we can take a single location to look at a timeseries

In [6]:
timeseries = ds.sel(latitude=52, longitude=5, method="nearest")

In [7]:
timeseries

<xarray.Dataset> Size: 7MB
Dimensions:    (time: 175320)
Coordinates:
    longitude  float32 4B 5.0
    latitude   float32 4B 52.0
  * time       (time) datetime64[ns] 1MB 2000-01-01 ... 2019-12-31T23:00:00
Data variables:
    d2m        (time) float32 701kB dask.array<chunksize=(2,), meta=np.ndarray>
    mslhf      (time) float32 701kB dask.array<chunksize=(2,), meta=np.ndarray>
    msshf      (time) float32 701kB dask.array<chunksize=(2,), meta=np.ndarray>
    sp         (time) float32 701kB dask.array<chunksize=(2,), meta=np.ndarray>
    ssr        (time) float32 701kB dask.array<chunksize=(2,), meta=np.ndarray>
    str        (time) float32 701kB dask.array<chunksize=(2,), meta=np.ndarray>
    t2m        (time) float32 701kB dask.array<chunksize=(2,), meta=np.ndarray>
    tp         (time) float32 701kB dask.array<chunksize=(2,), meta=np.ndarray>
Attributes:
    Conventions:  CF-1.6
    history:      2024-05-08 02:39:23 GMT by grib_to_netcdf-2.28.1: /opt/ecmw...

However, loading all timesteps is quite slow. As a demonstration we load the first time step into memory:

In [8]:
timeseries.isel(time=0).compute()

<xarray.Dataset> Size: 48B
Dimensions:    ()
Coordinates:
    longitude  float32 4B 5.0
    latitude   float32 4B 52.0
    time       datetime64[ns] 8B 2000-01-01
Data variables:
    d2m        float32 4B 277.6
    mslhf      float32 4B -0.6322
    msshf      float32 4B 7.206
    sp         float32 4B 1.467e+05
    ssr        float32 4B 0.0
    str        float32 4B -1.848e+04
    t2m        float32 4B 278.0
    tp         float32 4B 2.794e-06
Attributes:
    Conventions:  CF-1.6
    history:      2024-05-08 02:39:23 GMT by grib_to_netcdf-2.28.1: /opt/ecmw...